In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image

c:\Python\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
import tensorflow as tf
import keras

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)


c:\Python\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


TensorFlow version: 2.20.0
Keras version: 3.13.0


In [2]:
#params

IMAGE_SIZE = 256
BATCH_SIZE = 32
CHANNELS = 3
EPOCHS = 10
DATASET_PATH = "FinalDataSet" 

In [3]:
#Laading dataset
dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

Found 5239 files belonging to 20 classes.


In [4]:
class_names = dataset.class_names
num_classes = len(class_names)

In [5]:
print("Class Names:", class_names)
print("Number of Classes:", num_classes)

Class Names: ['burger', 'butter_naan', 'chai', 'chapati', 'chole_bhature', 'dal_makhani', 'dhokla', 'fried_rice', 'idli', 'jalebi', 'kaathi_rolls', 'kadai_paneer', 'kulfi', 'masala_dosa', 'momos', 'paani_puri', 'pakode', 'pav_bhaji', 'pizza', 'samosa']
Number of Classes: 20


In [6]:
# ===============================
# 4. SPLIT DATASET (80-10-10)
def split_dataset(ds, train_split=0.8, val_split=0.1, test_split=0.1, shuffle=True, shuffle_size=1000):
    ds_size = len(ds)
    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=12)
    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)
    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(train_size).skip(val_size)
    return train_ds, val_ds, test_ds

train_ds, val_ds, test_ds = split_dataset(dataset)


In [7]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)


In [8]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)


In [9]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))


In [10]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze base

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

model.summary()

C:\Users\Babu Menon\AppData\Local\Temp\ipykernel_17640\42455705.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(
c:\Python\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,424,532 (9.25 MB)

 Trainable params: 166,548 (650.58 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
#7. COMPILE MODEL
# ===============================
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [12]:
# ===============================
# 8. TRAIN MODEL
# ===============================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 461s 3s/step - accuracy: 0.5525 - loss: 1.5233 - val_accuracy: 0.8516 - val_loss: 0.5574
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 237s 2s/step - accuracy: 0.7927 - loss: 0.6994 - val_accuracy: 0.8867 - val_loss: 0.4005
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 226s 2s/step - accuracy: 0.8362 - loss: 0.5360 - val_accuracy: 0.9141 - val_loss: 0.3101
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 235s 2s/step - accuracy: 0.8573 - loss: 0.4634 - val_accuracy: 0.9180 - val_loss: 0.2813
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 227s 2s/step - accuracy: 0.8723 - loss: 0.4033 - val_accuracy: 0.9160 - val_loss: 0.2496
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 225s 2s/step - accuracy: 0.8991 - loss: 0.3395 - val_accuracy: 0.9199 - val_loss: 0.2321
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 243s 2s/step - accuracy: 0.8998 - loss: 0.3145 - val_accuracy: 0.9219 - val_loss: 0.2221
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 234s 2s/step - accuracy: 0.9056 - loss: 0.2926 - val_accu

In [13]:
# 9. EVALUATE MODEL
# ===============================
test_loss, test_accuracy = model.evaluate(test_ds)
print("Test Accuracy:", test_accuracy)


17/17 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.9577 - loss: 0.1577
Test Accuracy: 0.9577205777168274


In [14]:
for images, labels in train_ds.take(1):
    img = tf.expand_dims(images[0], axis=0)
    pred = model.predict(img)

    print("Actual    :", class_names[labels[0]])
    print("Predicted :", class_names[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Actual    : pakode
Predicted : pakode


In [15]:
model.save("food_mobilenet_model.h5")
with open("class_names.txt", "w") as f:
    for name in class_names:
        f.write(name + "\n")
print("✅ Model and class names saved")

✅ Model and class names saved


In [16]:
model.save("food_cnn_model.h5")
print("✅ Model saved as food_cnn_model.h5")

✅ Model saved as food_cnn_model.h5


In [17]:
with open("class_names.txt", "w") as f:
    for name in class_names:
        f.write(name + "\n")

print("✅ Class names saved")

✅ Class names saved
